The following is a workflow for segmenting and tracking cells from live cell imaging data. The segmentation is carried out using cellpose segmentation models (Stringer et al., 2021), and then the masked objects generated are tracked using btrack.

Some considerations while using the workflow -
1. When running on Colab, use T4 GPU for segmentation models.
Saved segmentation files can then be tracked on CPU.
2. Upload .tif files without channel data for segmentation. Segmentation is done in greyscale.
3. Segmented files require to be merged across different channels, if multi-channel data is being used, for a given time point. They must then be merged across all time points tracking using btrack. This exercise needs to be done externally.
5. Final outputs for tracking are .h5 files. Some information for their interpretation is provided below.

In [ ]:
!pip install cellpose

In [ ]:
!pip install "btrack==0.6.5" #Built in 2025. Requires older version of btrack.

In [ ]:
import h5py, matplotlib.pyplot as plt, numpy as np, os, re, shutil, zipfile, btrack
from google.colab import files
from natsort import natsorted
from pathlib import Path
from PIL import Image
from skimage.io import imread
from tqdm import trange

from cellpose import models, core, io, plot
from btrack import datasets #Import all necessary dependencies.

Ensure that files are numbered in ascending order, according to time point progression and do not contain the term "_masks" in the file name.
Suffix "_masks" is added to object mask files as an output from cellpose.

Upload .tif files to local directory and download Cellpose model(s).

In [ ]:
uploaded = files.upload() #You may alternatively mount your drive here as uploads can take time.

model = models.CellposeModel(gpu=True) # Calls cellpose model.
files = natsorted([f for f in Path(".").glob("*"+".tif") if "_masks" not in f.name]) # Sorts .tif files in ascending order.
print(f"{len(files)} files sorted.") # Displays number of files.

Images are segmented based on the called cellpose model. Information for different models is provided on their website.
Segmentation is done in greyscale and in sorted order.

Output appears as segmented panels. The mask .tif files are saved to the local directory.

In [ ]:
for i in trange(len(files)):
    f = files[i]
    img = io.imread(f) #Loop through and read images.
    # np.zeros_like(img)[:, :] = img[:, :] <- Flattens image. May be used if image has multi-channel data.
    masks, flows, styles = model.eval(img, batch_size=32, flow_threshold=0.4,
                                      cellprob_threshold=0.0, normalize={"tile_norm_blocksize": 0})
    plot.show_segmentation(plt.figure(figsize = (24,10)), np.zeros_like(img), masks, flows[0]) # Run cellpose model.
    plt.show() # Displays segmented image panels.
    io.imsave(Path(".") / (f.stem + "_masks" + ".tif"), masks) #Segmented images are saved as mask .tifs.

Downloads all masks in directory as a zip file, if required.

In [ ]:
os.makedirs('./masks', exist_ok=True)
for k in os.listdir('.'):
    item_path = os.path.join('.', k)
    if os.path.isfile(item_path) and "_masks" in k:
        shutil.move(item_path, os.path.join('./masks', k)) #Creates separate folder for masks within contents.

matching_files = [f for f in os.listdir('./masks') if 'masks' in f] #Searches for mask files.
with zipfile.ZipFile('Cell_Imaging_Masks.zip', 'w') as zipf:
    for file in matching_files:
        zipf.write(os.path.join('./masks', file)) #Combine masks to one zip file.

from google.colab import files as colab_files
colab_files.download('Cell_Imaging_Masks.zip') #Patch fix for zip download. Can be ignored for manual download.

If segmentation was done across different channels, files are to be merged for a given time point, and then across multiple time points, as indicated earlier. Required to be done externally.

Run the compiled .tif through btrack. Adjust tracking parameters as required.

In [ ]:
combined_tif = files.upload() #Re-upload the tif file here.

image = imread('Masks_combine.tif')
objects = btrack.utils.segmentation_to_objects(image, properties=('area',)) # Currently only tracks object area. Other properties may be used too.
with btrack.BayesianTracker() as tracker:
  tracker.configure(datasets.cell_config()) # Define tracker behaviour with JSON files. Leave as is for default.
  tracker.append(objects)
  tracker.volume = ((0, 1200), (0, 1600)) #Limit tracker to 2D
  tracker.track_interactive(step_size=100)
  tracker.optimize() #Optimizes path tracking of objects
  tracker.export('tracks.h5', obj_type='obj_type_1')
  tracks = tracker.tracks #Receive tracked properties as a list in the console.

In [ ]:
matching_files = [fname for fname in os.listdir('.') if 'track' in fname]
for fname in matching_files:
    print(fname) #Find and print track files.

colab_files.download('tracks.h5') #Download .h5 file.

# Print the file structure
def print_structure(name, obj):
    if isinstance(obj, h5py.Group):
        print(f"[Group]   {name}")
    elif isinstance(obj, h5py.Dataset):
        print(f"[Dataset] {name} — shape: {obj.shape}, dtype: {obj.dtype}") # Print file structure for information.

with h5py.File('tracks.h5', 'r') as f:
    f.visititems(print_structure)

with h5py.File('tracks.h5', 'r') as f:
    dset = f['tracks/obj_type_1/LBEPR']   # View any given subtype under the data.
    print("Shape:", dset.shape)
    print("Dtype:", dset.dtype)
    usable = np.array(dset)     # Convert to Numpy array.

print(usable, type(usable))

If multiple merged tif files are being run in the same session, remember to download, files as you go or rename file name in tracker.export('your_file_name.h5', ...) to prevent rewriting of previously collated data.

The output for the second part is the different data types in the file. They consist of - objects and tracks. Multiple data types called datasets (arrays) are present under each group.

Default settings give the datasets -
Objects -
1. coords - Coordinates of the segmented cells (All frames collated together)
2. labels - Numerical labels for each cell object.
3. map - Maps frame number to previous datasets.
4. properties - Properties set record by tracker. Default to segmentation area.
Tracks -
1. LBEPR - Label ID, Birth, End, Parent ID, Reserved(?)
2. dummies - Data added to correct missing tracks.
3. fates - Fate of a given track (0-ongoing, 1-terminated, 2-divided)
4. map - Mapping of ID to tracking movement.
5. tracks - Tracking movement.

From the output, take the path of your desired dataset and paste in f['your_dataset']. This gathers the shape and datatype of the dataset. Also convert to Numpy for further operations.

Export multiple datasets (arrays), as required.

(GPLv3 License).